# Goal: Understand the dimensions of the gradient tensorblocks

In [1]:
from ase.io import read
import numpy as np
from featomic import SoapPowerSpectrum, SphericalExpansion, SphericalExpansionByPair, NeighborList

systems = read("/Users/alin62/Downloads/dataset (2).xyz", ":")
len(systems)

40

In [2]:
# Just look at a single system
systems = [systems[0]]

In [3]:
HYPER_PARAMETERS = {
    "cutoff": {
        "radius": 5.0,
        "smoothing": {"type": "ShiftedCosine", "width": 0.5},
    },
    "density": {
        "type": "Gaussian",
        "width": 0.3,
    },
    "basis": {
        "type": "TensorProduct",
        "max_angular": 4,
        "radial": {"type": "Gto", "max_radial": 6},
    },
}

calculator = SoapPowerSpectrum(**HYPER_PARAMETERS)
se_calculator = SphericalExpansion(**HYPER_PARAMETERS)
sebp_calculator = SphericalExpansionByPair(**HYPER_PARAMETERS)
nl_calculator = NeighborList(cutoff=5.0, full_neighbor_list=True)

In [22]:
descriptor = calculator.compute(systems, gradients=["positions"])
se_descriptor = se_calculator.compute(systems, gradients=["positions"])
sebp_descriptor = sebp_calculator.compute(systems, gradients=["positions"])
nl = nl_calculator.compute(systems)

In [ ]:
import metatensor
sebp_descriptor = metatensor.rename_dimension(sebp_descriptor, axis="keys", old="o3_lambda", new="angular_component")
sebp_descriptor = metatensor.permute_dimensions(sebp_descriptor, axis="keys", dimensions_indexes=[1,0,2,3])
sebp_descriptor.block(1).gradien

TensorMap with 80 blocks
keys: o3_sigma  angular_component  first_atom_type  second_atom_type
         1              0                 1                1
         1              1                 1                1
                                   ...
         1              3                 8                8
         1              4                 8                8

In [28]:
sebp_descriptor.block(1).samples

Labels(
    system  first_atom  second_atom  cell_shift_a  cell_shift_b  cell_shift_c
      0         12          12            1             0             0
      0         12          12            -1            0             0
      0         12          13            1             0             1
      0         13          12            -1            0             -1
      0         12          13            0             -1            1
      0         13          12            0             1             -1
      0         12          13            0             0             1
      0         13          12            0             0             -1
      0         12          13            1             -1            1
      0         13          12            -1            1             -1
      0         12          13            -1            0             1
      0         13          12            1             0             -1
      0         12          14            -1 

In [30]:
sebp_descriptor.block(1).gradient("positions").samples

Labels(
    sample  system  atom
      0       0      12
      1       0      12
      2       0      12
      2       0      13
      3       0      13
      3       0      12
      4       0      12
      4       0      13
      5       0      13
      5       0      12
      6       0      12
      6       0      13
      7       0      13
      7       0      12
      8       0      12
      8       0      13
      9       0      13
      9       0      12
      10      0      12
      10      0      13
      11      0      13
      11      0      12
      12      0      12
      12      0      14
      13      0      14
      13      0      12
      14      0      12
      14      0      14
      15      0      14
      15      0      12
      16      0      12
      16      0      14
      17      0      14
      17      0      12
      18      0      12
      18      0      14
      19      0      14
      19      0      12
      20      0      12
      20      0      14
      2

In [209]:
block_i = 1
display(descriptor)
display(se_descriptor)
display(sebp_descriptor)
display(nl)
display("******CONTENTS******")
display(descriptor.block(block_i))
display(descriptor.block(block_i).gradient("positions"))
display("*******Spherical Expansion******")
display(se_descriptor.block(block_i))
display(se_descriptor.block(block_i).gradient("positions"))
display("******Spherical Expansion By Pair******")
display(sebp_descriptor.block(block_i))
display(sebp_descriptor.block(block_i).gradient("positions"))
display("******Neighbor List******")
display(nl.block(block_i))

TensorMap with 40 blocks
keys: center_type  neighbor_1_type  neighbor_2_type
           1              1                1
           1              1                6
                          ...
           8              7                8
           8              8                8

TensorMap with 80 blocks
keys: o3_lambda  o3_sigma  center_type  neighbor_type
          0         1           1             1
          1         1           1             1
                           ...
          3         1           8             8
          4         1           8             8

TensorMap with 80 blocks
keys: o3_lambda  o3_sigma  first_atom_type  second_atom_type
          0         1             1                1
          1         1             1                1
                               ...
          3         1             8                8
          4         1             8                8

TensorMap with 16 blocks
keys: first_atom_type  second_atom_type
             1                1
             1                6
                    ...
             8                7
             8                8

'******CONTENTS******'

...

'******Spherical Expansion By Pair******'

TensorBlock
    samples (172): ['system', 'first_atom', 'second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c']
    components (3): ['o3_mu']
    properties (7): ['n']
    gradients: ['positions']

Gradient TensorBlock ('positions')
    samples (328): ['sample', 'system', 'atom']
    components (3, 3): ['xyz', 'o3_mu']
    properties (7): ['n']
    gradients: None

'******Neighbor List******'

TensorBlock
    samples (92): ['system', 'first_atom', 'second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c']
    components (3): ['pair_xyz']
    properties (1): ['distance']
    gradients: None

In [192]:
descriptor.block(1).gradient("positions").samples

Labels(
    sample  system  atom
      0       0      4
      0       0      5
      0       0      6
      0       0      7
      0       0      12
      0       0      13
      0       0      14
      0       0      15
      0       0      16
      0       0      17
      0       0      18
      0       0      19
      1       0      4
      1       0      5
      1       0      6
      1       0      7
      1       0      12
      1       0      13
      1       0      14
      1       0      15
      1       0      16
      1       0      17
      1       0      18
      1       0      19
      2       0      4
      2       0      5
      2       0      6
      2       0      7
      2       0      12
      2       0      13
      2       0      14
      2       0      15
      2       0      16
      2       0      17
      2       0      18
      2       0      19
      3       0      4
      3       0      5
      3       0      6
      3       0      7
      3       0      12

In [245]:
display(sebp_descriptor)
display(sebp_descriptor.block(2))
display(sebp_descriptor.block(2).gradient("positions"))


display(sebp_descriptor.block(2).samples)
display(sebp_descriptor.block(2).gradient("positions").samples)

TensorMap with 80 blocks
keys: o3_lambda  o3_sigma  first_atom_type  second_atom_type
          0         1             1                1
          1         1             1                1
                               ...
          3         1             8                8
          4         1             8                8

TensorBlock
    samples (172): ['system', 'first_atom', 'second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c']
    components (5): ['o3_mu']
    properties (7): ['n']
    gradients: ['positions']

Gradient TensorBlock ('positions')
    samples (328): ['sample', 'system', 'atom']
    components (3, 5): ['xyz', 'o3_mu']
    properties (7): ['n']
    gradients: None

Labels(
    system  first_atom  second_atom  cell_shift_a  cell_shift_b  cell_shift_c
      0         12          12            1             0             0
      0         12          12            -1            0             0
      0         12          13            1             0             1
      0         13          12            -1            0             -1
      0         12          13            0             -1            1
      0         13          12            0             1             -1
      0         12          13            0             0             1
      0         13          12            0             0             -1
      0         12          13            1             -1            1
      0         13          12            -1            1             -1
      0         12          13            -1            0             1
      0         13          12            1             0             -1
      0         12          14            -1 

Labels(
    sample  system  atom
      0       0      12
      1       0      12
      2       0      12
      2       0      13
      3       0      13
      3       0      12
      4       0      12
      4       0      13
      5       0      13
      5       0      12
      6       0      12
      6       0      13
      7       0      13
      7       0      12
      8       0      12
      8       0      13
      9       0      13
      9       0      12
      10      0      12
      10      0      13
      11      0      13
      11      0      12
      12      0      12
      12      0      14
      13      0      14
      13      0      12
      14      0      12
      14      0      14
      15      0      14
      15      0      12
      16      0      12
      16      0      14
      17      0      14
      17      0      12
      18      0      12
      18      0      14
      19      0      14
      19      0      12
      20      0      12
      20      0      14
      2

In [236]:
sebp_descriptor.block(0).values.squeeze()

ExternalCpuArray([[-1.24572564e-03,  3.12420864e-03, -5.23172129e-03,
                   ...,  7.07090896e-02,  2.76175247e-02,
                    8.32806435e-03],
                  [-1.24572564e-03,  3.12420864e-03, -5.23172129e-03,
                   ...,  7.07090896e-02,  2.76175247e-02,
                    8.32806435e-03],
                  [ 4.50640755e-04, -8.54442377e-04,  1.45493150e-03,
                   ..., -1.05826056e-02,  3.44062942e-02,
                    2.18960439e-02],
                  ...,
                  [ 9.09461763e-01, -2.66052998e-01,  3.46712128e-02,
                   ...,  7.21870384e-02, -3.38972952e-02,
                    6.77177614e-03],
                  [ 9.09461763e-01, -2.66052998e-01,  3.46712128e-02,
                   ...,  7.21870384e-02, -3.38972952e-02,
                    6.77177614e-03],
                  [ 9.09461763e-01, -2.66052998e-01,  3.46712128e-02,
                   ...,  7.21870384e-02, -3.38972952e-02,
                    6.77

In [244]:
sebp_descriptor.block(0).gradient("positions").values.squeeze()[3]

ExternalCpuArray([[-0.00214632,  0.00418477, -0.00907964,  0.01841173,
                    0.00592892, -0.11605136, -0.0462867 ],
                  [-0.00157706,  0.00307486, -0.00667148,  0.01352844,
                    0.00435641, -0.08527142, -0.03401022],
                  [-0.00079286,  0.00154587, -0.00335406,  0.00680137,
                    0.00219017, -0.04286987, -0.0170985 ]])

In [336]:
sebp_descriptor
labels = sebp_descriptor.keys
labels

Labels(
    o3_lambda  o3_sigma  first_atom_type  second_atom_type
        0         1             1                1
        1         1             1                1
        2         1             1                1
        3         1             1                1
        4         1             1                1
        0         1             1                6
        1         1             1                6
        2         1             1                6
        3         1             1                6
        4         1             1                6
        0         1             1                7
        1         1             1                7
        2         1             1                7
        3         1             1                7
        4         1             1                7
        0         1             1                8
        1         1             1                8
        2         1             1                8
        3      

In [349]:
labels.names[0] = 'angular_component'
sebp_descriptor.set_info('o3_lambda', 'angular_component')

AttributeError: 'TensorMap' object has no attribute 'set_info'

In [334]:
se_descriptor

TensorMap with 80 blocks
keys: o3_lambda  o3_sigma  center_type  neighbor_type
          0         1           1             1
          1         1           1             1
                           ...
          3         1           8             8
          4         1           8             8

In [277]:
sebp_descriptor.block(4)

TensorBlock
    samples (172): ['system', 'first_atom', 'second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c']
    components (9): ['o3_mu']
    properties (7): ['n']
    gradients: ['positions']

In [284]:
import metatensor
ex = metatensor.sum_over_samples_block(sebp_descriptor.block(0), ['second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c'])
ex.values

array([[[ 0.90612717, -0.24450564,  0.14088404,  0.24691609,
          0.78568092,  0.62950157,  0.29347647]],

       [[ 0.90607781, -0.2438795 ,  0.141134  ,  0.24822768,
          0.78497006,  0.62700994,  0.29272137]],

       [[ 0.92399233, -0.29249457,  0.33034112,  0.29465863,
          0.6304863 ,  0.3828306 ,  0.17020351]],

       [[ 0.92402289, -0.29188168,  0.33169567,  0.29444081,
          0.63007443,  0.38213823,  0.17005719]],

       [[ 0.90823192, -0.22939602,  0.16238175,  0.0962183 ,
          0.80756239,  0.52634418,  0.21384405]],

       [[ 0.90824575, -0.23033804,  0.16224683,  0.09685774,
          0.80846205,  0.52639357,  0.21365928]],

       [[ 0.92814789, -0.27783942,  0.34918177,  0.09564588,
          0.62426238,  0.56826035,  0.23951068]],

       [[ 0.92825394, -0.27887955,  0.35015297,  0.09578885,
          0.62416393,  0.56957568,  0.23992274]]])

In [283]:
se_descriptor.block(0).values

ExternalCpuArray([[[ 0.90612717, -0.24450564,  0.14088404,  0.24691609,
                     0.78568092,  0.62950157,  0.29347647]],

                  [[ 0.90607781, -0.2438795 ,  0.141134  ,  0.24822768,
                     0.78497006,  0.62700994,  0.29272137]],

                  [[ 0.92399233, -0.29249457,  0.33034112,  0.29465863,
                     0.6304863 ,  0.3828306 ,  0.17020351]],

                  [[ 0.92402289, -0.29188168,  0.33169567,  0.29444081,
                     0.63007443,  0.38213823,  0.17005719]],

                  [[ 0.90823192, -0.22939602,  0.16238175,  0.0962183 ,
                     0.80756239,  0.52634418,  0.21384405]],

                  [[ 0.90824575, -0.23033804,  0.16224683,  0.09685774,
                     0.80846205,  0.52639357,  0.21365928]],

                  [[ 0.92814789, -0.27783942,  0.34918177,  0.09564588,
                     0.62426238,  0.56826035,  0.23951068]],

                  [[ 0.92825394, -0.27887955,  0.35015297,  0.

In [294]:
sebp_descriptor.block(1).samples.values.shape

(172, 6)

In [301]:
nl_block = nl.block(0)
sample_col = np.tile(np.arange(len(nl_block.samples)), 2)
system_col = np.tile(nl_block.samples['system'], 2)
atom_col = np.hstack((nl_block.samples['first_atom'], nl_block.samples['second_atom']))
sign_col = np.hstack((np.ones(len(nl_block.samples), dtype=np.int8), -np.ones(len(nl_block.samples), dtype=np.int8)))
samples = metatensor.Labels(
    names=['sample', 'system', 'atom', 'sign'],
    values=np.vstack((sample_col, system_col, atom_col, sign_col)).T
)
samples
len(samples)//2

172

In [316]:
display(sebp_descriptor.block(1).samples)
display(sebp_descriptor.block(1).gradient("positions").samples)
# display(sebp_descriptor.block(1).gradient("positions").samples)
display(se_descriptor.block(1).samples)
display(se_descriptor.block(1).gradient("positions").samples)

Labels(
    system  first_atom  second_atom  cell_shift_a  cell_shift_b  cell_shift_c
      0         12          12            1             0             0
      0         12          12            -1            0             0
      0         12          13            1             0             1
      0         13          12            -1            0             -1
      0         12          13            0             -1            1
      0         13          12            0             1             -1
      0         12          13            0             0             1
      0         13          12            0             0             -1
      0         12          13            1             -1            1
      0         13          12            -1            1             -1
      0         12          13            -1            0             1
      0         13          12            1             0             -1
      0         12          14            -1 

Labels(
    sample  system  atom
      0       0      12
      1       0      12
      2       0      12
      2       0      13
      3       0      13
      3       0      12
      4       0      12
      4       0      13
      5       0      13
      5       0      12
      6       0      12
      6       0      13
      7       0      13
      7       0      12
      8       0      12
      8       0      13
      9       0      13
      9       0      12
      10      0      12
      10      0      13
      11      0      13
      11      0      12
      12      0      12
      12      0      14
      13      0      14
      13      0      12
      14      0      12
      14      0      14
      15      0      14
      15      0      12
      16      0      12
      16      0      14
      17      0      14
      17      0      12
      18      0      12
      18      0      14
      19      0      14
      19      0      12
      20      0      12
      20      0      14
      2

Labels(
    system  atom
      0      12
      0      13
      0      14
      0      15
      0      16
      0      17
      0      18
      0      19
)

Labels(
    sample  system  atom
      0       0      12
      0       0      13
      0       0      14
      0       0      15
      0       0      16
      0       0      17
      0       0      18
      0       0      19
      1       0      12
      1       0      13
      1       0      14
      1       0      15
      1       0      16
      1       0      17
      1       0      18
      1       0      19
      2       0      12
      2       0      13
      2       0      14
      2       0      15
      2       0      16
      2       0      17
      2       0      18
      2       0      19
      3       0      12
      3       0      13
      3       0      14
      3       0      15
      3       0      16
      3       0      17
      3       0      18
      3       0      19
      4       0      12
      4       0      13
      4       0      14
      4       0      15
      4       0      16
      4       0      17
      4       0      18
      4       0      19
      5

In [319]:
# Inspect sample 2 (i.e. ) of sebp (nonzero derivative) and assert symmetry of x-derivative (and therefore y derivative and z derivative)
assert np.allclose(sebp_descriptor.block(1).gradient("positions").values[2][0], -sebp_descriptor.block(1).gradient("positions").values[3][0])

# Now see relationship between sample 2 of sebp and sample 2
print(sebp_descriptor.block(1).gradient("positions").values[2][0])
print()

[[ 0.00217319 -0.00423328  0.00912006 -0.01816908 -0.00751991  0.11912979
   0.04883524]
 [ 0.00109256 -0.00212826  0.00458507 -0.00913443 -0.00378061  0.05989203
   0.02455172]
 [ 0.00279096 -0.0054453   0.01187401 -0.02437532 -0.0063183   0.14940341
   0.05836192]]



In [333]:
import metatensor
ex = metatensor.sum_over_samples_block(sebp_descriptor.block(0), sample_names=['second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c'])
display(ex)
display(ex.gradient("positions"))
display(ex.gradient('positions').samples)
display(ex.gradient('positions').values[0][0])
display(se_descriptor.block(0).gradient('positions').values[0][0])
ex.gradient('positions') == se_descriptor.block(0).gradient('positions')

TensorBlock
    samples (8): ['system', 'first_atom']
    components (1): ['o3_mu']
    properties (7): ['n']
    gradients: ['positions']

Gradient TensorBlock ('positions')
    samples (64): ['sample', 'system', 'atom']
    components (3, 1): ['xyz', 'o3_mu']
    properties (7): ['n']
    gradients: None

Labels(
    sample  system  atom
      0       0      12
      0       0      13
      0       0      14
      0       0      15
      0       0      16
      0       0      17
      0       0      18
      0       0      19
      1       0      12
      1       0      13
      1       0      14
      1       0      15
      1       0      16
      1       0      17
      1       0      18
      1       0      19
      2       0      12
      2       0      13
      2       0      14
      2       0      15
      2       0      16
      2       0      17
      2       0      18
      2       0      19
      3       0      12
      3       0      13
      3       0      14
      3       0      15
      3       0      16
      3       0      17
      3       0      18
      3       0      19
      4       0      12
      4       0      13
      4       0      14
      4       0      15
      4       0      16
      4       0      17
      4       0      18
      4       0      19
      5

array([[-0.00918104, -0.03603928, -0.08931318,  0.20906118, -0.07220429,
        -0.17157813,  0.0171304 ]])

ExternalCpuArray([[-0.00918104, -0.03603928, -0.08931318,  0.20906118,
                   -0.07220429, -0.17157813,  0.0171304 ]])

True

In [322]:
se_descriptor.block(1).gradient("positions").values[0][0]

ExternalCpuArray([[ 0.0067324 ,  0.05625124,  0.11568934, -0.0482737 ,
                   -0.08728248, -0.03158788, -0.01060922],
                  [-0.0250151 ,  0.12846165, -0.09779761, -0.19096878,
                    0.38491384, -0.14765708, -0.04317508],
                  [-0.01008039,  0.02424209, -0.1267325 , -0.0017618 ,
                    0.08117664,  0.11143337,  0.02627259]])

In [191]:
nl.block(0)

TensorBlock
    samples (9134): ['system', 'first_atom', 'second_atom', 'cell_shift_a', 'cell_shift_b', 'cell_shift_c']
    components (3): ['pair_xyz']
    properties (1): ['distance']
    gradients: None

In [177]:
18108/9554

1.8953317981997069

# Conclusion: we see that the spherical expansion 

In [172]:
display("******Descriptor Shape for l=2******")
display(se_descriptor.block(2))
display(se_descriptor.block(2).components)
display(se_descriptor.block(2).values.shape)
display("******Gradient Shape for l=2******")
display(se_descriptor.block(2).gradient("positions"))
display(se_descriptor.block(2).gradient("positions").components)
display(se_descriptor.block(2).gradient("positions").values.shape)

'******Descriptor Shape for l=2******'

TensorBlock
    samples (420): ['system', 'atom']
    components (5): ['o3_mu']
    properties (7): ['n']
    gradients: ['positions']

[Labels(
     o3_mu
      -2
      -1
       0
       1
       2
 )]

(420, 5, 7)

'******Gradient Shape for l=2******'

Gradient TensorBlock ('positions')
    samples (4968): ['sample', 'system', 'atom']
    components (3, 5): ['xyz', 'o3_mu']
    properties (7): ['n']
    gradients: None

[Labels(
     xyz
      0
      1
      2
 ),
 Labels(
     o3_mu
      -2
      -1
       0
       1
       2
 )]

(4968, 3, 5, 7)

In [ ]:
# Looking at system 0, hydrogen-hydrogen-hydrogen triplets, we see that there are 8^2=64 entries.
descriptor.block(0).gradient("positions").samples

Labels(
    sample  system  atom
      0       0      12
      0       0      13
      0       0      14
      0       0      15
      0       0      16
      0       0      17
      0       0      18
      0       0      19
      1       0      12
      1       0      13
      1       0      14
      1       0      15
      1       0      16
      1       0      17
      1       0      18
      1       0      19
      2       0      12
      2       0      13
      2       0      14
      2       0      15
      2       0      16
      2       0      17
      2       0      18
      2       0      19
      3       0      12
      3       0      13
      3       0      14
      3       0      15
      3       0      16
      3       0      17
      3       0      18
      3       0      19
      4       0      12
      4       0      13
      4       0      14
      4       0      15
      4       0      16
      4       0      17
      4       0      18
      4       0      19
      5

In [156]:
nl.block(0).samples

Labels(
    system  first_atom  second_atom  cell_shift_a  cell_shift_b  cell_shift_c
      0         12          12            1             0             0
      0         12          13            1             0             1
      0         12          13            0             -1            1
      0         12          13            0             0             1
      0         12          13            1             -1            1
      0         12          13            -1            0             1
      0         12          14            -1            0             0
      0         12          14            0             0             0
      0         12          14            1             0             0
      0         12          14            -1            1             0
      0         12          14            0             1             0
      0         12          15            0             -1            1
      0         12          15            1       

In [125]:
descriptor.block(0).gradient('positions')

Gradient TensorBlock ('positions')
    samples (4968): ['sample', 'system', 'atom']
    components (3): ['xyz']
    properties (245): ['l', 'n_1', 'n_2']
    gradients: None

In [126]:
descriptor.block(0)

TensorBlock
    samples (420): ['system', 'atom']
    components (): []
    properties (245): ['l', 'n_1', 'n_2']
    gradients: ['positions']

In [127]:
descriptor.block(0).samples

Labels(
    system  atom
      0      12
      0      13
      0      14
      0      15
      0      16
      0      17
      0      18
      0      19
      1      12
      1      13
      1      14
      1      15
      1      16
      1      17
      1      18
      1      19
      2      12
      2      13
      2      14
      2      15
      2      16
      2      17
      2      18
      2      19
      3      12
      3      13
      3      14
      3      15
      3      16
      3      17
      3      18
      3      19
      4      12
      4      13
      4      14
      4      15
      4      16
      4      17
      4      18
      4      19
      5      12
      5      13
      5      14
      5      15
      5      16
      5      17
      5      18
      5      19
      6      12
      6      13
      6      14
      6      15
      6      16
      6      17
      6      18
      6      19
      7      12
      7      13
      7      14
      7      15
      7      16

In [128]:
descriptor.block(0).gradient('positions').samples

Labels(
    sample  system  atom
      0       0      12
      0       0      13
      0       0      14
      0       0      15
      0       0      16
      0       0      17
      0       0      18
      0       0      19
      1       0      12
      1       0      13
      1       0      14
      1       0      15
      1       0      16
      1       0      17
      1       0      18
      1       0      19
      2       0      12
      2       0      13
      2       0      14
      2       0      15
      2       0      16
      2       0      17
      2       0      18
      2       0      19
      3       0      12
      3       0      13
      3       0      14
      3       0      15
      3       0      16
      3       0      17
      3       0      18
      3       0      19
      4       0      12
      4       0      13
      4       0      14
      4       0      15
      4       0      16
      4       0      17
      4       0      18
      4       0      19
      5

In [129]:
# Where 4968 comes from is: for each of the 40 systems: sum the square of the length of each system.
samples = descriptor.block(0).samples
tot = 0
for i in range(1):
    tot += (samples['system'] == i).sum()**2
print(tot)
    

64


In [59]:
# look at neighborlist
from featomic.calculators import NeighborList
nl = NeighborList(cutoff=5.0, full_neighbor_list=False, self_pairs=False).compute(systems[-1])
nl.block(0).samples

Labels(
    system  first_atom  second_atom  cell_shift_a  cell_shift_b  cell_shift_c
      0         18          19            -1            -1            1
      0         18          19            -1            0             1
      0         18          19            0             0             1
      0         18          20            -1            0             0
      0         18          21            0             0             1
      0         18          21            1             0             1
      0         18          22            -1            -1            0
      0         18          23            0             0             1
      0         18          26            0             0             0
      0         18          26            -1            0             0
      0         18          27            0             0             1
      0         18          28            -1            0             0
      0         18          28            0       

In [67]:
nl.block(0).values.reshape(-1, 3)

ExternalCpuArray([[-1.41365151, -4.57670293,  0.19281923],
                  [-3.38845927,  0.97226601,  0.19281923],
                  [ 1.99904073,  0.97226601,  0.19281923],
                  [-0.779459  ,  0.4076881 , -2.68809   ],
                  [-2.63657927,  0.80953601,  2.79063623],
                  [ 2.75092073,  0.80953601,  2.79063623],
                  [-0.72680125, -1.77682094, -3.87645   ],
                  [-0.37597327, -2.47090099,  3.83878623],
                  [ 1.951241  ,  0.670738  , -3.01644   ],
                  [-3.436259  ,  0.670738  , -3.01644   ],
                  [-0.04544927,  0.73937601,  3.23444623],
                  [-2.723139  , -0.889912  , -0.59729   ],
                  [ 2.664361  , -0.889912  , -0.59729   ],
                  [ 0.68955325,  4.65905694, -0.59729   ],
                  [-0.46135927,  2.09743601,  0.47202623],
                  [ 1.51344849, -3.45153293,  0.47202623],
                  [-2.84252425, -2.99326094, -1.78814   

In [81]:
from featomic.calculators import SortedDistances
sd = SortedDistances(cutoff=5.0, max_neighbors=len(systems[-1]), separate_neighbor_types=True).compute(systems[-1])
np.array(sd.block(0).values)

array([[2.19884037, 2.23128759, 2.29738984, 2.7409457 , 2.82835531,
        2.87184927, 2.92646284, 3.17083263, 3.31819001, 3.50133797,
        3.53045842, 3.65460548, 3.79821203, 3.92358884, 4.00132029,
        4.03599163, 4.15361449, 4.2635    , 4.31670677, 4.32575974,
        4.49855531, 4.58071908, 4.62132834, 4.7403857 , 4.747531  ,
        4.79393362, 4.84451796, 5.        , 5.        , 5.        ,
        5.        , 5.        , 5.        , 5.        ],
       [2.12946721, 2.23128759, 2.54861536, 2.62423638, 2.70932792,
        2.71984048, 2.9108702 , 3.14831359, 3.22374763, 3.37061188,
        3.53045842, 3.67228429, 3.83527497, 3.92750158, 3.99142299,
        4.04208449, 4.30803993, 4.33638713, 4.36333305, 4.45912026,
        4.46627905, 4.52564343, 4.70129959, 4.79393362, 4.84077832,
        4.98698852, 4.99759747, 5.        , 5.        , 5.        ,
        5.        , 5.        , 5.        , 5.        ],
       [2.1206882 , 2.48737856, 2.68990617, 2.69172668, 2.76292082,
  

In [8]:
print("before: ", len(descriptor.keys))

descriptor = descriptor.keys_to_samples("center_type")
descriptor = descriptor.keys_to_properties(["neighbor_1_type", "neighbor_2_type"])
print("after: ", len(descriptor.keys))


before:  40
after:  1
